In [13]:
from pathlib import Path
import pandas as pd
import numpy as np
import torch

BASE_PATH = Path(
    "/content/drive/MyDrive/ECs Master Folder/Research/"
    "Kidney Thermal Property MFNN/Data/raw"
)

# nested dictionaries
# outer dictionary (text as key) -> dictionaries as the value
# that dictionary value has key-value pairs
EXPERIMENTS = {
    "porcine_conductivity": {
        "file": "porcine_raw.csv",
        "target": "conductivity_mean_W_mK",
        "uncertainty": "conductivity_EU95_W_mK",
    },
    "porcine_diffusivity": {
        "file": "porcine_raw.csv",
        "target": "diffusivity_mean_mm2_s",
        "uncertainty": "diffusivity_EU95_mm2_s",
    },
    "bovine_conductivity": {
        "file": "bovine_raw.csv",
        "target": "conductivity_mean_W_mK",
        "uncertainty": "conductivity_EU95_W_mK"
    },
    "bovine_diffusivity": {
        "file": "bovine_raw.csv",
        "target": "diffusivity_mean_mm2_s",
        "uncertainty": "diffusivity_EU95_mm2_s",
    },
}

TEMP_COL = "temperature_C"
HF_SIZES = [3, 5, 7, 9]
SEEDS = range(20)

In [14]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
# returns a pandas dataframe w 3 columns of the hi-fi data
def load_experiment(config): # config = one inner dictionary
  df = pd.read_csv(BASE_PATH / config["file"]) # opens csv file

  required_columns = [
      TEMP_COL,
      config["target"],
      config["uncertainty"]
  ]

  missing_columns = []

  # adds column
  for column in required_columns:
    if column not in df.columns:
        missing_columns.append(column) # stores a variable w/ missing column names

  if missing_columns: # non-empty list -> True, empty list -> False
    raise ValueError(
        f"Missing columns: {missing_columns}\n"
        f"Available columns: {list(df.columns)}"
    )

  return (
      df[required_columns]
      .dropna() # deletes rows w/ missing values
      .sort_values(TEMP_COL) # low -> high
      .reset_index(drop=True) # resets row indices
  )

In [16]:
for experiment_name, config in EXPERIMENTS.items():
  data = load_experiment(config) # config = value

  print(experiment_name) # key
  print("Number of HF points:", len(data))
  display(data)

porcine_conductivity
Number of HF points: 11


,temperature_C,conductivity_mean_W_mK,conductivity_EU95_W_mK
0,23.9,0.549,0.045
1,35.4,0.559,0.050
2,41.5,0.573,0.050
3,46.2,0.573,0.030
4,56.7,0.571,0.033
5,60.0,0.560,0.022
6,70.1,0.536,0.017
7,76.4,0.528,0.026
8,82.3,0.527,0.080
9,86.6,0.596,0.058


porcine_diffusivity
Number of HF points: 11


,temperature_C,diffusivity_mean_mm2_s,diffusivity_EU95_mm2_s
0,23.9,0.155,0.019
1,35.4,0.157,0.011
2,41.5,0.161,0.009
3,46.2,0.163,0.011
4,56.7,0.161,0.007
5,60.0,0.160,0.008
6,70.1,0.165,0.011
7,76.4,0.171,0.014
8,82.3,0.178,0.016
9,86.6,0.194,0.033


bovine_conductivity
Number of HF points: 10


,temperature_C,conductivity_mean_W_mK,conductivity_EU95_W_mK
0,23.2,0.528,0.066
1,30.2,0.548,0.030
2,36.9,0.551,0.091
3,40.9,0.528,0.180
4,48.8,0.541,0.129
5,56.4,0.562,0.167
6,69.5,0.595,0.183
7,75.8,0.590,0.164
8,81.1,0.550,0.191
9,90.0,0.703,0.226


bovine_diffusivity
Number of HF points: 10


,temperature_C,diffusivity_mean_mm2_s,diffusivity_EU95_mm2_s
0,23.2,0.151,0.020
1,30.2,0.153,0.030
2,36.9,0.157,0.035
3,40.9,0.157,0.032
4,48.8,0.160,0.043
5,56.4,0.155,0.015
6,69.5,0.162,0.029
7,75.8,0.169,0.031
8,81.1,0.160,0.009
9,90.0,0.183,0.006


In [ ]:
def split_hf_data(data, n_hf, seed):
  rng = np.random.default_rng(seed)

  all_indices = np.arange(len(data)) # creates all row positions in a list

  # same row can't be selected twice
  # picking n_hf indices
  train_indices = rng.choice(
      all_indices,
      size=n_hf,
      replace=False
  )

  # selects all indices that were not put into train_indices
  test_indices = np.setdiff1d(
      all_indices,
      train_indices
  )

  # selects all rows in pandas df w/ those row #s, renumbers rows starting from 0
  train_data = data.iloc[train_indices].reset_index(drop=True)
  test_data = data.iloc[test_indices].reset_index(drop=True)

  return train_data, test_data